# Day 26: A/B Testing and Continuous Improvement

'I think this change is better' is a guess until you measure it. This notebook splits simulated traffic between two real versions of Day 22's RagService, compares them on more than one metric, checks whether any difference is real or just luck, and plans a safe rollout. Everything here is seeded, so every run produces the same numbers, and the whole notebook finishes in a few seconds.

## The two variants

Variant A is the default config (top_k=3). Variant B changes exactly one thing: top_k=5.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
from variants import VARIANT_A, VARIANT_B
for v in (VARIANT_A, VARIANT_B):
    print(v.name, ':', v.description)


## Sticky traffic routing

The same user id always lands in the same group, computed by hashing the id rather than rolling a fresh random choice per request.

In [ ]:
from traffic_router import assign_variant
user = 'user-00042'
print([assign_variant(user) for _ in range(5)])

user_ids = [f'user-{i:05d}' for i in range(1000)]
counts = {'A': 0, 'B': 0}
for u in user_ids:
    counts[assign_variant(u)] += 1
print(counts)


## Run the experiment

50 simulated users, routed to A or B, each getting a real RagService response and simulated feedback based on whether the top result was actually relevant.

In [ ]:
import ab_experiment
ab_experiment.main()


## Compare metrics

Helpfulness rate, Day 8's real precision/recall/NDCG, average latency, and average sources returned, all in one table.

In [ ]:
import metric_comparison
metric_comparison.main()


## Is the difference real?

A two-proportion z-test, implemented with only math.erfc. The same percentage gap is inconclusive at 30 users per arm and conclusive at 500.

In [ ]:
import significance_check
significance_check.main()


## How many users do I need?

Sample size depends heavily on the smallest effect worth detecting, chosen before the experiment starts, not after looking at early results.

In [ ]:
import sample_size_planner
sample_size_planner.main()


## Safe rollout

Staged rollout at 1%, 10%, 50%, 100%, with guardrails on error rate and p95 latency checked at every stage, separate from the main helpfulness goal.

In [ ]:
import safe_rollout
safe_rollout.main()


## Experiment log

Every experiment's hypothesis, sample sizes, rates, p-value, and decision, saved to a temporary JSON Lines file, read back, then deleted.

In [ ]:
import experiment_log
experiment_log.main()


## Key takeaways

- A change is a guess until it is measured against a real, randomly assigned comparison group.
- Sticky assignment keeps one user's experience consistent and keeps the experiment's result attributable to the actual change.
- A win on one metric can hide a loss on another, which is why helpfulness, retrieval quality, latency, and source count are all checked together.
- A raw percentage difference means nothing without knowing whether the sample size was large enough to trust it.
- Sample size should be planned before an experiment starts, based on the smallest effect worth catching, not decided by peeking at early results.
- A safe rollout checks guardrails, not just the goal metric, at every stage, and rolls back the moment one breaks.
- Every experiment decision belongs in a written, auditable record.